# PasarPulse-MM: Paper B1 ARIMA Baseline

**Methodological subtitle:** A leakage-safe rolling-origin ARIMA baseline for province–commodity–market-level horticultural price forecasting

**Authors and team:** Aditya Akbar · Taqiyudin Miftah Adn · Muhammad Faris El Hakim · Team PasarPulse

**Scope of this executed notebook:** the paper's B1 ARIMA baseline only. The paper's B0 seasonal-naïve baseline, the previous local naïve implementation, the proposed boosting baseline, and later multimodal/graph architectures are deliberately not used here.

## Table of Contents

0. Title Page  
Abstract  
1. Introduction and Research Questions  
2. Related Work and Methodological Positioning  
3. Dependencies and Reproducibility  
4. Data and Exploratory Analysis  
5. Feature Engineering  
6. Model Architecture  
7. Validation Strategy  
8. Ablation Study  
9. Explainable AI  
10. Inference-Time and Cost Analysis  
11. Final Training and Submission  
12. Discussion and Research Questions  
13. Limitations  
14. Conclusion  
References

## Abstract

This notebook trains and evaluates the paper-defined B1 ARIMA baseline on the audited PIHPS daily price panel. The unit of analysis is province × commodity × market level × day, and the forecast horizons are seven and fourteen days. Each eligible series receives a fixed ARIMA(1,1,1) fit at declared rolling origins; no future labels are used for model or order selection. The executed outputs include data-integrity checks, fold-level and pooled metrics, 95% forecast intervals, out-of-fold predictions, and fit audits. Later architectures are declared for the research protocol but are intentionally not executed in this baseline notebook.

# 1. Introduction and Research Questions

## 1.1 Problem Context

PasarPulse-MM targets early warning for horticultural price shocks in Indonesia across producer, wholesale, and retail market levels. The paper defines seven- and fourteen-day forecasting tasks for bird's-eye chili, red/curly chili, and shallots.

## 1.2 Main Challenges

The data are asynchronous and revised across sources, coverage differs by province and market level, and price series can contain missing days. The baseline therefore uses only the versioned price panel and respects the eligibility audit.

## 1.3 Research Questions

- **RQ1:** How accurately does the paper's B1 ARIMA baseline forecast prices at seven and fourteen days?
- **RQ2:** How stable are its errors across rolling origins and eligible series?
- **RQ3:** How often does the baseline identify the paper's ≥10% price-shock label?
- **RQ4:** What performance and coverage improvements remain for calendar, weather, satellite, and graph architectures?

# 2. Related Work and Methodological Positioning

## 2.1 Relevant Methods

The proposal positions ARIMA as B1, after B0 seasonal naïve and before the price-only TFT (B2). The proposed system is a missing-modality-aware multimodal dynamic graph model.

## 2.2 Alternative Approaches

The paper also names price-plus-calendar, price-plus-weather, all-modalities-without-graph, and static-graph variants. Those are future comparison points and are not trained here.

## 2.3 Justification of Selected Approach

This notebook selects B1 because it is a trainable, paper-specified baseline that isolates univariate price dynamics. The paper does not publish an ARIMA order-search protocol, so ARIMA(1,1,1) is frozen before scoring and is reported as a reproducible operational choice rather than presented as the paper's undisclosed exact hyperparameter.

# 3. Dependencies and Reproducibility

## 3.1 Imports and Random Seed

The ARIMA dependency is `statsmodels>=0.14`. A seed is recorded for notebook-level reproducibility, although fixed ARIMA fitting itself is deterministic for a fixed environment and data.

## 3.2 Execution Modes

`EXECUTION_MODE="full"` runs every eligible series and declared origin. Set `EXECUTION_MODE="smoke"` only for a quick development check; the checked-in executed notebook uses full mode.

## 3.3 Portable Path Discovery

The notebook discovers the bundle root from the processed PIHPS artifact, so it can run from the local bundle or the A100 copy without hard-coded user-specific paths.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
import platform
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels

SEED = 20260804
np.random.seed(SEED)
EXECUTION_MODE = os.environ.get("PASARPULSE_EXECUTION_MODE", "full").lower()
if EXECUTION_MODE not in {"full", "smoke"}:
    raise ValueError("PASARPULSE_EXECUTION_MODE must be full or smoke")

def discover_root() -> Path:
    candidates = []
    configured = os.environ.get("PASARPULSE_ROOT")
    if configured:
        candidates.append(Path(configured).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    for candidate in candidates:
        if (candidate / "data" / "processed" / "price_daily.parquet").exists():
            return candidate
    raise FileNotFoundError(
        "Could not discover the bundle root from data/processed/price_daily.parquet"
    )

ROOT = discover_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print({
    "root": str(ROOT),
    "execution_mode": EXECUTION_MODE,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "statsmodels": statsmodels.__version__,
    "seed": SEED,
})

{'root': '<PROJECT_ROOT>', 'execution_mode': 'full', 'python': '3.10.4', 'pandas': '1.5.3', 'numpy': '1.24.1', 'statsmodels': '0.14.4', 'seed': 20260804}


# 4. Data and Exploratory Analysis

## 4.1 Data Loading and Integrity Checks

In [2]:
price_path = ROOT / "data" / "processed" / "price_daily.parquet"
coverage_path = ROOT / "data" / "processed" / "price_coverage_audit.csv"
prices = pd.read_parquet(price_path) if price_path.exists() else pd.read_csv(
    ROOT / "data" / "processed" / "price_daily.csv"
)
coverage_audit = pd.read_csv(coverage_path)
prices["date"] = pd.to_datetime(prices["date"], errors="coerce").dt.normalize()
prices["price_idr_per_kg"] = pd.to_numeric(prices["price_idr_per_kg"], errors="coerce")
prices = prices.dropna(subset=["date", "price_idr_per_kg"]).copy()
key_columns = ["date", "province_code", "commodity_code", "market_level"]
duplicate_rows = int(prices.duplicated(key_columns).sum())
assert duplicate_rows == 0, f"duplicate canonical panel rows: {duplicate_rows}"
assert prices["price_idr_per_kg"].gt(0).all(), "non-positive prices found"
assert prices["commodity_code"].isin(["BM", "CRM", "CMK"]).all()
print({
    "price_rows": int(len(prices)),
    "coverage_rows": int(len(coverage_audit)),
    "duplicate_panel_rows": duplicate_rows,
    "date_min": str(prices["date"].min().date()),
    "date_max": str(prices["date"].max().date()),
    "units": sorted(prices["price_unit"].dropna().unique().tolist()),
    "sources": sorted(prices["source"].dropna().unique().tolist()),
})

{'price_rows': 181177, 'coverage_rows': 282, 'duplicate_panel_rows': 0, 'date_min': '2024-02-01', 'date_max': '2026-07-31', 'units': ['IDR/kg'], 'sources': ['PIHPS']}


## 4.2 Dataset Summary

In [3]:
series_columns = ["province_code", "commodity_code", "market_level"]
series_count = prices[series_columns].drop_duplicates().shape[0]
print("eligible series:", int(coverage_audit["eligible"].astype(str).str.lower().isin(["true", "1", "yes"]).sum()))
print("observed series:", series_count)
print("rows by market level:")
print(prices["market_level"].value_counts().sort_index().to_string())
print("rows by commodity:")
print(prices["commodity_code"].value_counts().sort_index().to_string())

eligible series: 279
observed series: 282
rows by market level:
producer     52218
retail       64425
wholesale    64534
rows by commodity:
BM     58930
CMK    63794
CRM    58453


## 4.3 Target and Entity Heterogeneity

In [4]:
entity_summary = (
    prices.groupby(series_columns, dropna=False)["price_idr_per_kg"]
    .agg(observed_days="count", median_price="median", mean_price="mean", std_price="std")
    .reset_index()
)
print(entity_summary.describe(include="all").transpose().to_string())
entity_summary.head(10)

                count unique     top freq          mean           std           min           25%           50%           75%           max
province_code     282     34      52    9           NaN           NaN           NaN           NaN           NaN           NaN           NaN
commodity_code    282      3     CMK   99           NaN           NaN           NaN           NaN           NaN           NaN           NaN
market_level      282      3  retail  100           NaN           NaN           NaN           NaN           NaN           NaN           NaN
observed_days   282.0    NaN     NaN  NaN    642.471631     46.612061         147.0         648.0         651.0         652.0         652.0
median_price    282.0    NaN     NaN  NaN  42311.702128  13269.289844       19000.0       32500.0       40000.0       48350.0       95000.0
mean_price      282.0    NaN     NaN  NaN  44080.103522  13874.348238  18288.343558  34537.193252   41433.51227  51238.561781  98706.896552
std_price       282.

,province_code,commodity_code,market_level,observed_days,median_price,mean_price,std_price
0,11,BM,producer,652,30400.0,31171.702454,6058.804312
1,11,BM,retail,652,39100.0,41173.236196,8087.441239
2,11,BM,wholesale,652,33150.0,34409.202454,7068.357236
3,11,CMK,producer,652,30850.0,31272.469325,10713.117942
4,11,CMK,retail,652,42300.0,45283.358896,16257.992100
5,11,CMK,wholesale,652,32000.0,34602.760736,13179.473639
6,11,CRM,wholesale,648,34000.0,35675.925926,10160.416910
7,12,BM,producer,626,24800.0,24350.079872,6131.979451
8,12,BM,retail,652,39750.0,40991.411043,7005.651866
9,12,BM,wholesale,652,31850.0,33006.748466,6505.388326


## 4.4 Temporal Coverage

In [5]:
coverage_view = coverage_audit[[
    "market_level", "eligible", "coverage", "months_available", "observed_days"
]].copy()
coverage_view["eligible"] = coverage_view["eligible"].astype(str).str.lower().isin(["true", "1", "yes"])
print(coverage_view.groupby("market_level").agg(
    series=("eligible", "size"), eligible=("eligible", "sum"), median_coverage=("coverage", "median")
).to_string())

              series  eligible  median_coverage
market_level                                   
producer          82        81         0.996933
retail           100        99         0.997699
wholesale        100        99         1.000000


## 4.5 Missing Values and Anomalies

Missing observations are retained as missing on a daily index for ARIMA's state-space handling. Long-gap imputation is not performed, and evaluation targets are scored only when their observed target date exists.

In [6]:
missing_summary = prices.isna().sum().sort_values(ascending=False).head(10)
print("missing values in loaded canonical data:")
print(missing_summary.to_string())
print("coverage exclusions:")
print(coverage_audit.loc[~coverage_view["eligible"], ["province_name", "commodity_name", "market_level", "reason"]].to_string(index=False))

missing values in loaded canonical data:
date                    0
province_code           0
province_name           0
commodity_code          0
commodity_name          0
market_level            0
price_idr_per_kg        0
price_unit              0
source                  0
source_location_name    0
coverage exclusions:
 province_name       commodity_name market_level              reason
Kepulauan Riau    Cabai Rawit Merah       retail insufficient_months
Kepulauan Riau    Cabai Rawit Merah    wholesale insufficient_months
  Maluku Utara Cabai Merah Keriting     producer insufficient_months


# 5. Feature Engineering

## 5.1 Raw Features

The B1 model uses only `price_idr_per_kg` and the observation date index.

## 5.2 Aggregated Features

None are used by B1.

## 5.3 Temporal Features

None are used by B1; calendar features are reserved for the paper's B3 variant.

## 5.4 Static and Entity Features

None are used by the individual ARIMA experts.

## 5.5 Historical Features

The ARIMA(1,1,1) state-space model internally represents autoregressive and differenced historical price dynamics.

## 5.6 Leakage and Availability Audit

In [7]:
leakage_audit = {
    "target": "price_idr_per_kg",
    "features": ["historical_price_observations_only"],
    "future_label_used_for_fit": False,
    "future_label_used_for_order_selection": False,
    "long_gap_imputation": False,
    "evaluation_protocol": "rolling_origin",
    "eligibility_source": "price_coverage_audit.csv",
    "model": "B1_ARIMA(1,1,1)",
}
print(json.dumps(leakage_audit, indent=2))

{
  "target": "price_idr_per_kg",
  "features": [
    "historical_price_observations_only"
  ],
  "future_label_used_for_fit": false,
  "future_label_used_for_order_selection": false,
  "long_gap_imputation": false,
  "evaluation_protocol": "rolling_origin",
  "eligibility_source": "price_coverage_audit.csv",
  "model": "B1_ARIMA(1,1,1)"
}


# 6. Model Architecture

## 6.1 Baselines

### Executed model: paper B1 ARIMA

One individual ARIMA(1,1,1) model is fitted per eligible province × commodity × market-level series at each declared rolling origin. The paper names ARIMA as B1 but does not specify an order-search procedure; the order is therefore predeclared and frozen. This cell deliberately does not call the previous `last_price`/`seasonal_lag` implementation or any proposed global boosting model.

In [8]:
from pasarpulse.tools.arima_baseline import ARIMAConfig, make_rolling_origins, run_arima_baseline

START_DATE = str(prices["date"].min().date())
END_DATE = str(prices["date"].max().date())
EVALUATION_START = str((pd.Timestamp(END_DATE) - pd.Timedelta(days=180)).date())
ARIMA_CONFIG = ARIMAConfig(
    order=(1, 1, 1),
    min_train_observations=365,
    max_horizon_days=14,
    shock_threshold=0.10,
    confidence_alpha=0.05,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
origins = make_rolling_origins(
    START_DATE,
    END_DATE,
    min_train_days=ARIMA_CONFIG.min_train_observations,
    max_horizon_days=ARIMA_CONFIG.max_horizon_days,
    evaluation_start=EVALUATION_START,
    # PIHPS observations are scheduled weekdays. Select origins from
    # the common observed calendar and require both calendar targets
    # to be observable, then advance by 20 scheduled observations.
    stride_days=20,
    available_dates=prices["date"].drop_duplicates().tolist(),
)
if EXECUTION_MODE == "smoke":
    origins = origins[:2]
print("declared origins:", [str(origin.date()) for origin in origins])
started_at = time.perf_counter()
arima_metrics, arima_predictions, arima_fit_audit, arima_metadata = run_arima_baseline(
    prices,
    coverage_audit,
    start_date=START_DATE,
    end_date=END_DATE,
    origins=origins,
    config=ARIMA_CONFIG,
)
runtime_seconds = time.perf_counter() - started_at
arima_metadata["runtime_seconds"] = runtime_seconds
print(json.dumps(arima_metadata, indent=2, default=str))
print(arima_metrics.to_string(index=False))

declared origins: ['2026-02-02', '2026-03-02', '2026-03-30', '2026-04-27', '2026-05-25', '2026-06-22', '2026-07-17']
{
  "protocol": "paper_B1_ARIMA_rolling_origin",
  "model": "B1_ARIMA(1, 1, 1)",
  "arima_order": [
    1,
    1,
    1
  ],
  "source_date_range": [
    "2024-02-01",
    "2026-07-31"
  ],
  "declared_origins": [
    "2026-02-02",
    "2026-03-02",
    "2026-03-30",
    "2026-04-27",
    "2026-05-25",
    "2026-06-22",
    "2026-07-17"
  ],
  "eligible_series_available": 279,
  "series_seen": 279,
  "series_scored": 279,
  "fit_attempts": 1953,
  "successful_fits": 1944,
  "prediction_rows": 3880,
  "horizons_days": [
    7,
    14
  ],
  "min_train_observations": 365,
  "shock_threshold": 0.1,
  "confidence_alpha": 0.05,
  "leakage_guard": "each fit uses only price observations on or before its forecast origin",
  "runtime_seconds": 316.7710566930473
}
            model arima_order  horizon_days  rows_scored  series_scored  folds_scored  mae_idr_per_kg  rmse_idr_per_kg

## 6.2 Target Transformation

**Reserved for a later architecture.** B1 uses the raw IDR/kg target; no log or Box–Cox transformation is executed.

## 6.3 Individual Experts or Models

**Reserved for later architectures.** B1's per-series ARIMA fits are already complete in §6.1.

## 6.4 Global Combination

**Reserved for later architectures.** No global ensemble or stacking is executed.

## 6.5 Local Adaptation

**Reserved for later architectures.** No local/global adaptation layer is executed.

## 6.6 Predeclared Domain Rule

**Reserved for later architectures.** No post-hoc prediction correction or domain rule is applied to B1.

# 7. Validation Strategy

## 7.1 Split Design

Validation is chronological. Each fit sees only observations on or before its forecast origin, and the seven- and fourteen-day labels are read only after the origin.

## 7.2 Fold Construction

The executed folds use the most recent 180 days of the common price window, with fixed 20-scheduled-observation origin spacing and a minimum of 365 observed training values per series. Origins are chosen only from dates for which both seven- and fourteen-calendar-day targets are present in the common PIHPS calendar.

## 7.3 Out-of-Fold Predictions

In [9]:
OUTPUT_DIR = ROOT / "reports" / "paper_b1_arima"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def write_table(frame: pd.DataFrame, stem: str) -> None:
    frame.to_csv(TABLE_DIR / f"{stem}.csv", index=False)
    try:
        frame.to_parquet(TABLE_DIR / f"{stem}.parquet", index=False)
    except (ImportError, ModuleNotFoundError):
        pass

write_table(arima_predictions, "b1_arima_oof_predictions")
write_table(arima_fit_audit, "b1_arima_fit_audit")
write_table(arima_metrics, "b1_arima_pooled_metrics")

fold_rows = []
for (fold_id, origin, horizon), group in arima_predictions.groupby(
    ["fold_id", "forecast_origin", "horizon_days"], sort=True
):
    actual = group["actual_price_idr_per_kg"].astype(float)
    predicted = group["predicted_price_idr_per_kg"].astype(float)
    error = predicted - actual
    fold_rows.append({
        "fold_id": int(fold_id),
        "forecast_origin": str(pd.Timestamp(origin).date()),
        "horizon_days": int(horizon),
        "rows_scored": int(len(group)),
        "series_scored": int(group["series_id"].nunique()),
        "mae_idr_per_kg": float(error.abs().mean()),
        "rmse_idr_per_kg": float(np.sqrt(np.mean(np.square(error)))),
        "mean_error_idr_per_kg": float(error.mean()),
        "actual_shock_rate": float(group["actual_shock"].mean()),
        "predicted_shock_rate": float(group["predicted_shock"].mean()),
        "shock_recall": float(
            ((group["actual_shock"] == 1) & (group["predicted_shock"] == 1)).sum()
            / max((group["actual_shock"] == 1).sum(), 1)
        ),
    })
fold_metrics = pd.DataFrame(fold_rows)
write_table(fold_metrics, "b1_arima_fold_metrics")
with (OUTPUT_DIR / "b1_arima_metadata.json").open("w", encoding="utf-8") as handle:
    json.dump({**arima_metadata, "leakage_audit": leakage_audit}, handle, indent=2, default=str)
print("saved outputs:", OUTPUT_DIR)

saved outputs: <PROJECT_ROOT>/reports/paper_b1_arima


## 7.4 Fold-Level Metrics

In [10]:
print(fold_metrics.to_string(index=False))

 fold_id forecast_origin  horizon_days  rows_scored  series_scored  mae_idr_per_kg  rmse_idr_per_kg  mean_error_idr_per_kg  actual_shock_rate  predicted_shock_rate  shock_recall
       1      2026-02-02             7          279            279     3642.144187      6633.215892           -2783.585591           0.265233              0.000000      0.000000
       1      2026-02-02            14          279            279     5371.384904      9513.991661           -4615.531167           0.358423              0.000000      0.000000
       2      2026-03-02             7          279            279     2660.537289      4906.657499            -441.886828           0.100358              0.003584      0.000000
       2      2026-03-02            14          279            279     4864.194706      9054.099648           -3176.931052           0.261649              0.003584      0.013699
       3      2026-03-30             7          279            279     4014.958725      8018.234750           

## 7.5 Pooled Metric

In [11]:
print(arima_metrics.to_string(index=False))

            model arima_order  horizon_days  rows_scored  series_scored  folds_scored  mae_idr_per_kg  rmse_idr_per_kg  mean_error_idr_per_kg  smape_percent  interval_coverage_95  actual_shock_rate  predicted_shock_rate  shock_precision  shock_recall  shock_f1  true_positive  false_positive  false_negative
B1_ARIMA(1, 1, 1)   (1, 1, 1)             7         1941            279             7     3731.688788      6997.664305              39.142772       7.660800              0.951571           0.131376              0.002061             0.25      0.003922  0.007722              1               3             254
B1_ARIMA(1, 1, 1)   (1, 1, 1)            14         1939            279             7     5812.425134      9812.915115             -87.906958      11.754634              0.947368           0.206292              0.003094             0.50      0.007500  0.014778              3               3             397


## 7.6 Validation Visualization and Interpretation

In [12]:
if not arima_predictions.empty:
    example_id = arima_predictions["series_id"].value_counts().index[0]
    example = arima_predictions[arima_predictions["series_id"] == example_id].sort_values(
        ["target_date", "horizon_days"]
    )
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(example["target_date"], example["actual_price_idr_per_kg"], "o-", label="actual")
    for horizon, color in [(7, "tab:orange"), (14, "tab:green")]:
        view = example[example["horizon_days"] == horizon]
        ax.plot(view["target_date"], view["predicted_price_idr_per_kg"], "o--", color=color, label=f"ARIMA h={horizon}")
    ax.set_title(f"B1 ARIMA rolling-origin example: {example_id}")
    ax.set_ylabel("IDR/kg")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "b1_arima_example_forecasts.png", dpi=160)
    plt.show()

# 8. Ablation Study

## 8.1 Individual Components
**Not executed.** No alternate architecture or feature component is being claimed yet.

## 8.2 Progressive System Variants
**Not executed.** The next planned variants are the paper's B3–B6 systems.

## 8.3 Contribution Analysis
**Not executed.** Contribution claims require the later model runs.

# 9. Explainable AI

## 9.1 Explanation Method
**Not executed.** The paper's perturbation-validated XAI is reserved for the multimodal system.

## 9.2 Additive Reconstruction Check
**Not executed.**

## 9.3 Global Explanation
**Not executed.**

## 9.4 Local Explanation
**Not executed.**

## 9.5 Model or Routing Audit
**Not executed.** The present audit is limited to ARIMA fit status and forecast coverage.

# 10. Inference-Time and Cost Analysis

## 10.1 Runtime Benchmark
The executed runtime is recorded in `b1_arima_metadata.json`; a deployment benchmark is not claimed yet.

## 10.2 Memory and Model Size
**Not executed.** Per-series fitted model persistence is reserved for final training.

## 10.3 Deployment Requirements
The baseline requires the versioned price panel, `statsmodels`, and the eligibility audit. It does not require weather, satellite, graph, or LLM services.

# 11. Final Training and Submission

## 11.1 Final Fitting
**Not executed.** This notebook is the rolling-origin baseline evaluation artifact.

## 11.2 Prediction
Out-of-fold predictions are exported for research evaluation only.

## 11.3 Submission Validation
**Not executed.** No competition submission schema was supplied to this baseline notebook.

## 11.4 File Export
Evaluation tables and figures are exported under `reports/paper_b1_arima/`.

# 12. Discussion and Research Questions

## 12.1 Answer RQ1
**Pending model comparison.** The executed pooled metrics provide the B1 ARIMA answer for the two horizons.

## 12.2 Answer RQ2
**Pending interpretation.** Fold-level metrics are exported for stability analysis.

## 12.3 Answer RQ3
**Pending interpretation.** Shock precision, recall, and false negatives are included in pooled metrics.

## 12.4 Answer RQ4
**Pending later architectures.** Weather, satellite, and graph improvements cannot be claimed from this notebook.

# 13. Limitations

This is a univariate per-series ARIMA baseline. The selected order is a reproducibility decision because the paper does not specify its ARIMA order protocol. Missing target dates are not imputed for evaluation, and provinces/series failing the paper's quality contract are excluded from training and scoring.

# 14. Conclusion

This executed notebook establishes the paper's B1 ARIMA reference point with an auditable rolling-origin protocol. It intentionally stops model development at §6.1; later transformations, global combinations, local adaptation, domain rules, multimodal features, and dynamic graphs remain future work.

# References

1. PasarPulse-MM proposal, `semi-finals/paper/PasarPulse.pdf`, §3.4 and Table 5.
2. Bank Indonesia, PIHPS commodity price tables, used through the versioned acquisition artifacts in this bundle.